# Phase 2 — Data Coverage and Mask Review V4
Named human review, immutable originals, automatic per-image saves to Drive. No GPU required. Correct masks from the original image; model predictions are not displayed. New data are staged for review and never enter training automatically.

## 1. Setup

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, urllib.request, hashlib, importlib.util, json

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CameraTrap_Phase2')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shapely', 'imagehash', 'ipywidgets'], check=True)
CODE = Path('/content/data_review_v4_code'); CODE.mkdir(exist_ok=True)
CHECKS = {'workflow.py': 'de0dd96242ccb576210101e3d19999d40152eb933dac56f7d71a6bc102bb4d18', 'mask_editor.js': '440b3d4f22764567339cade0b892a03c7a304ab5f0b473a4ff7af333be0a3f95', 'collect.py': '571de9e2d77a90bbf227706ea6d4868f44ff8bd7d66ea3105aad428746c124de'}
for name, expected in CHECKS.items():
    with urllib.request.urlopen('https://raw.githubusercontent.com/salehdev3/Capstone_5/phase-2/data_review_v4/'+name, timeout=60) as response:
        source = response.read()
    assert hashlib.sha256(source).hexdigest() == expected, name
    (CODE/name).write_bytes(source)
def load_module(name):
    spec = importlib.util.spec_from_file_location(name, CODE/(name+'.py'))
    module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module); return module
workflow = load_module('workflow')
collector = load_module('collect')
print('Ready — CPU only. No training starts in this notebook.')


## 2. Build inventory and mask queue
Audit 190 train/validation references; geometry findings come first. Source images and labels must already be in Drive. Existing six training-only removals are reused.

In [ ]:
REVIEW = workflow.initialize(ROOT)
from IPython.display import display
import pandas as pd
display(pd.read_csv(REVIEW/'inventory/species_coverage.csv'))
display(pd.read_csv(REVIEW/'queue.csv').head(10))


## 3. Review masks — 20 images per batch
Enter your name and observation. Select a polygon to drag points; Shift-click deletes a vertex. You may load a suggestion if available, but inspect all animals before confirming. Save verified only after reviewing the reference; choose Needs correction to defer. Repeat this cell for another batch. Close without saving ends the batch. A valid draft is preserved if Python rejects geometry.
If the original visible mask is correct despite a topology warning, restore the original and explicitly justify retaining that unchanged topology; exceptions remain recorded.


In [ ]:
for _ in range(20):
    record = workflow.review_next(REVIEW)
    if record is None:
        break
print('Batch ended. Saved reviews remain in Drive.')


## 4. Revisit deferred corrections (optional)

In [ ]:
workflow.review_flagged(REVIEW)


## 5. Save and download mask review
You may export partial progress. Status records actual verified counts. Original references are preserved.

In [ ]:
archive = workflow.export_review(REVIEW)
from google.colab import files
files.download(str(archive))


## 6. Plan extra training images (optional, no GPU)
Read the public CCT20 catalogue and find low-support species in unused events from training cameras. This does not expand validation/test coverage. Keep the printed availability summary; unavailable species require another data source.

In [ ]:
candidate_plan = collector.plan(ROOT, workflow)


## 7. Download and quarantine duplicates
Stage at most 150 planned images; this may take several minutes. No new training release is created.

In [ ]:
candidate_records = collector.download(ROOT, workflow)


## 8. Review one new training candidate
Check every shown species label and box against the photograph. Needs correction keeps the image out of verified candidates. Run again for the next image.

In [ ]:
collector.review_next(ROOT, workflow)


## 9. Export collection audit
Contains catalogue summary, staged records and actual team review records. It excludes images and the raw catalogue.

In [ ]:
import zipfile
from datetime import datetime, timezone
folder = ROOT/'data_review_v4/collection'
archive = ROOT/'data_review_v4'/('Collection_Review_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in folder.rglob('*.json'):
        if 'metadata' not in path.relative_to(folder).parts:
            z.write(path, path.relative_to(folder))
print('Saved:', archive)
from google.colab import files
files.download(str(archive))
